# Tutorial 1: Basic Graph Construction and Hyperdimensional Memory

This notebook introduces the core data structures of the **Spectral Memory Graph
Processor (SMGP)**: the `SpectralMemoryGraph` and the `HyperdimensionalMemory`
engine.

We will:

1. Create a knowledge graph and add nodes and edges.
2. Query similar nodes using HD vector similarity.
3. Verify factual claims against the graph.
4. Explore hyperdimensional memory operations (bind, unbind, bundle, similarity).

**References:**
- Kanerva, P. (2009). "Hyperdimensional Computing." *Cognitive Computation*, 1(2).
- Angles, R. & Gutierrez, C. (2008). "Survey of Graph Database Models." *ACM Computing Surveys*.

In [1]:
# Verify SMGP is installed and check the version
import smgp
print(f"SMGP version: {smgp.__version__}")

import numpy as np
np.set_printoptions(precision=4, suppress=True)

SMGP version: 0.1.0


## 1. Creating a Knowledge Graph

The `SpectralMemoryGraph` is a directed, multi-relational property graph where
every node carries a **hyperdimensional (HD) vector**. These vectors serve as
content-addressable keys, enabling O(1) approximate associative recall.

We create a graph with a smaller HD dimension (1000) for faster demonstration.

In [2]:
from smgp.core.graph import SpectralMemoryGraph

# Create a graph with HD vectors of dimension 1000 and a fixed seed
graph = SpectralMemoryGraph(hd_dim=1000, seed=42)
print(f"Created graph with HD dim = {graph.hd.dim}")
print(f"Nodes: {graph.num_nodes}, Edges: {graph.num_edges}")

Created graph with HD dim = 1000
Nodes: 0, Edges: 0


## 2. Adding Nodes and Edges

Each node is identified by a unique string ID, has a semantic label, and
optional properties. Edges are directed, typed relations between nodes.

Let us build a small knowledge graph about cities and countries.

In [3]:
# Add entity nodes
entities = [
    ("Paris",     "city",    {"country": "France",  "population": 2_161_000}),
    ("London",    "city",    {"country": "UK",      "population": 8_982_000}),
    ("Berlin",    "city",    {"country": "Germany", "population": 3_645_000}),
    ("France",    "country", {"continent": "Europe"}),
    ("UK",        "country", {"continent": "Europe"}),
    ("Germany",   "country", {"continent": "Europe"}),
    ("Europe",    "continent", {}),
]

for node_id, label, props in entities:
    graph.add_node(node_id, label=label, properties=props)

# Add typed edges (relations)
graph.add_edge("Paris",  "France",  "capital_of")
graph.add_edge("London", "UK",      "capital_of")
graph.add_edge("Berlin", "Germany", "capital_of")
graph.add_edge("France",  "Europe", "part_of")
graph.add_edge("UK",      "Europe", "part_of")
graph.add_edge("Germany", "Europe", "part_of")

print(f"Nodes: {graph.num_nodes}, Edges: {graph.num_edges}")
print(f"All nodes: {graph.nodes()}")

Nodes: 7, Edges: 6
All nodes: ['Paris', 'London', 'Berlin', 'France', 'UK', 'Germany', 'Europe']


## 3. Querying Similar Nodes

Each node has an HD vector. We can find the most similar nodes by cosine
similarity in the HD space. This is the core of content-addressable retrieval.

Let's query nodes similar to "Paris".

In [4]:
# Get the HD vector for Paris
paris_data = graph.get_node("Paris")
paris_vector = paris_data["vector"]
print(f"Paris HD vector shape: {paris_vector.shape}, dtype: {paris_vector.dtype}")
print(f"First 10 values: {paris_vector[:10]}")

# Find the 3 most similar nodes
similar = graph.query_similar(paris_vector, k=3)
print("\nTop-3 most similar nodes to 'Paris':")
for rank, (node_id, sim) in enumerate(similar, 1):
    node = graph.get_node(node_id)
    print(f"  {rank}. {node_id:10s} (sim={sim:.4f}, label={node['label']})")

Paris HD vector shape: (1000,), dtype: int8
First 10 values: [-1  1  1 -1 -1  1 -1  1 -1 -1]

Top-3 most similar nodes to 'Paris':
  1. Paris      (sim=1.0000, label=city)
  2. Europe     (sim=0.0620, label=continent)
  3. Berlin     (sim=0.0140, label=city)


## 4. Verifying Claims

The `ClaimVerifier` checks factual claims against the graph. It supports
direct triple format and natural language claims.

In [5]:
from smgp.reasoning.verifier import ClaimVerifier

verifier = ClaimVerifier(graph)

claims = [
    "Paris capital_of France",    # Direct triple — should be VERIFIED
    "London capital_of Germany",  # Wrong — should NOT be verified
    "Berlin is the capital of Germany",  # Natural language
]

for claim in claims:
    result = verifier.verify(claim)
    status = "VERIFIED" if result["verified"] else "NOT VERIFIED"
    print(f"  [{status}] '{claim}' (confidence={result['confidence']:.2f})")

  [VERIFIED] 'Paris capital_of France' (confidence=1.00)
  [NOT VERIFIED] 'London capital_of Germany' (confidence=0.00)
  [VERIFIED] 'Berlin is the capital of Germany' (confidence=0.80)


## 5. Hyperdimensional Memory Operations

The `HyperdimensionalMemory` engine provides the core algebra:

- **Generate**: Create random bipolar {-1, +1}^D vectors.
- **Bundle**: Superpose vectors via element-wise majority (approximate sum).
- **Bind/Unbind**: Associative pairing via element-wise multiplication.
- **Permute**: Cyclic shift for sequence order encoding.
- **Similarity**: Cosine similarity for approximate matching.

Let's explore these operations.

In [6]:
from smgp.core.hyperdim import HyperdimensionalMemory

hd = HyperdimensionalMemory(dim=10000, seed=42)
print(f"HD dimension: {hd.dim}")

# Generate 5 random bipolar vectors
vectors = hd.generate(5)
print(f"Generated vectors shape: {vectors.shape}")
print(f"Unique values: {np.unique(vectors)}")  # Should be {-1, 1}

HD dimension: 10000
Generated vectors shape: (5, 10000)
Unique values: [-1  1]


In [7]:
# Bind and unbind: associative pairing
role = vectors[0]    # e.g., "capital_of"
filler = vectors[1]  # e.g., "Paris"

bound = hd.bind(role, filler)
recovered = hd.unbind(bound, role)  # Should recover filler

# Check recovery quality
sim = hd.similarity(filler, recovered)
print(f"Bind/Unbind recovery similarity: {sim:.6f}")
print(f"Perfect recovery (should be 1.0 for bipolar vectors): {np.array_equal(filler, recovered)}")

Bind/Unbind recovery similarity: 1.000000
Perfect recovery (should be 1.0 for bipolar vectors): True


In [8]:
# Bundle: superpose multiple vectors into one
bundled = hd.bundle(vectors)
print(f"Bundled vector shape: {bundled.shape}")

# Check similarity of bundle to each constituent
print("Similarity of bundle to each constituent:")
for i, v in enumerate(vectors):
    s = hd.similarity(bundled, v)
    print(f"  vector {i}: similarity = {s:.4f}")

Bundled vector shape: (10000,)
Similarity of bundle to each constituent:
  vector 0: similarity = 0.3700
  vector 1: similarity = 0.3740
  vector 2: similarity = 0.3780
  vector 3: similarity = 0.3830
  vector 4: similarity = 0.3782


In [9]:
# Permute: cyclic shift for sequence order encoding
original = vectors[0]
shifted_1 = hd.permute(original, shifts=1)
shifted_5 = hd.permute(original, shifts=5)

# Original should be nearly orthogonal to its permutations
sim_orig_shift1 = hd.similarity(original, shifted_1)
sim_orig_shift5 = hd.similarity(original, shifted_5)
print(f"Original vs shift(1): {sim_orig_shift1:.4f}")
print(f"Original vs shift(5): {sim_orig_shift5:.4f}")
print("Permutations create near-orthogonal vectors for encoding position.")

Original vs shift(1): 0.0056
Original vs shift(5): -0.0048
Permutations create near-orthogonal vectors for encoding position.


## 6. Building a Larger Graph

HD vectors scale to large graphs. Each node's vector is drawn independently
from a high-dimensional space, ensuring low collision probability.
For D=10000, the expected similarity of two random vectors is ~0 with
standard deviation ~0.01.

In [10]:
# Build a larger graph with 100 nodes
big_graph = SpectralMemoryGraph(hd_dim=10000, seed=123)

for i in range(100):
    big_graph.add_node(f"node_{i}", label="concept", properties={"index": i})

# Chain edges
for i in range(99):
    big_graph.add_edge(f"node_{i}", f"node_{i+1}", "next")

# Some cross-links
for i in range(0, 100, 10):
    big_graph.add_edge(f"node_{i}", f"node_{(i+5) % 100}", "related")

print(f"Large graph: {big_graph.num_nodes} nodes, {big_graph.num_edges} edges")

# Query similarity
query_vec = big_graph.hd.generate(1)[0]
results = big_graph.query_similar(query_vec, k=5)
print("\nTop-5 most similar to a random query:")
for rank, (nid, sim) in enumerate(results, 1):
    print(f"  {rank}. {nid} (sim={sim:.4f})")

Large graph: 100 nodes, 109 edges

Top-5 most similar to a random query:
  1. node_80 (sim=0.0244)
  2. node_82 (sim=0.0192)
  3. node_12 (sim=0.0180)
  4. node_79 (sim=0.0170)
  5. node_4 (sim=0.0148)
